# MamayLM Fine-tuning Notebook

Fine-tune [MamayLM-Gemma-3-12B-IT](https://huggingface.co/INSAIT-Institute/MamayLM-Gemma-3-12B-IT-v1.0) for Ukrainian war euphemism detection.

**Features:**
- Iterative training — run N epochs at a time, backup weights, then continue
- Google Drive checkpoint backup after every iteration
- Resume training from the latest checkpoint
- Interactive testing of the model between iterations
- Supports LoRA, prompt-tuning, and full fine-tuning modes

## 1. Setup & Install Dependencies

In [ ]:
!pip install -q transformers accelerate peft datasets bitsandbytes
!pip install -q pandas openpyxl scikit-learn

## 2. Mount Google Drive & Configure Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Paths ─────────────────────────────────────────────────────────────────────
# Google Drive folder for backups (created automatically)
DRIVE_BACKUP_DIR = "/content/drive/MyDrive/mamaylm_checkpoints"
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)

# Local working directories (on Colab VM — fast but ephemeral)
LOCAL_OUTPUT_DIR = "/content/mamaylm_finetuned"

# Training data — upload these to Colab or place them in Drive
TRAIN_DATA_PATH = "PETs_Ukr_Train.xlsx"   # adjust if stored in Drive
TEST_DATA_PATH  = "PETs_Ukr_Test.xlsx"

print(f"Drive backup dir: {DRIVE_BACKUP_DIR}")
print(f"Local output dir: {LOCAL_OUTPUT_DIR}")

## 3. Upload Training Data

Run **one** of the two options below:
- **Option A** — upload from your computer (Colab file dialog)
- **Option B** — copy from Google Drive

In [ ]:
# ── Option A: upload via Colab ────────────────────────────────────────────────
from google.colab import files
uploaded = files.upload()  # select PETs_Ukr_Train.xlsx and PETs_Ukr_Test.xlsx

In [ ]:
# ── Option B: copy from Drive ─────────────────────────────────────────────────
# Uncomment and adjust the source path if your data lives in Drive:
# !cp "/content/drive/MyDrive/Euphemisms/PETs_Ukr_Train.xlsx" .
# !cp "/content/drive/MyDrive/Euphemisms/PETs_Ukr_Test.xlsx" .

## 4. Configuration

In [ ]:
# ── Training mode ─────────────────────────────────────────────────────────────
# Choose exactly one: "lora", "prompt_tuning", "full_finetune"
TRAINING_MODE = "lora"

# ── Iteration settings ────────────────────────────────────────────────────────
EPOCHS_PER_ITERATION = 1   # epochs to train in each iteration
TOTAL_ITERATIONS     = 3   # total number of iterations to run

# ── Model ──────────────────────────────────────────────────────────────────────
MODEL_NAME = "INSAIT-Institute/MamayLM-Gemma-3-12B-IT-v1.0"

# ── Hyperparameters ───────────────────────────────────────────────────────────
MAX_LENGTH = 512
BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
WARMUP_STEPS = 100

# Learning rates (per mode)
LEARNING_RATES = {
    "lora":          2e-4,
    "prompt_tuning": 3e-2,
    "full_finetune": 2e-5,
}

# LoRA-specific
LORA_RANK    = 8
LORA_ALPHA   = 16
LORA_DROPOUT = 0.1

# ── System prompt ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = "Напиши лише цифру 1, якщо вираз в кутових дужках є евфемізмом, інакше  0. "

print(f"Mode: {TRAINING_MODE}")
print(f"Iterations: {TOTAL_ITERATIONS} × {EPOCHS_PER_ITERATION} epoch(s) = {TOTAL_ITERATIONS * EPOCHS_PER_ITERATION} total epochs")
print(f"Learning rate: {LEARNING_RATES[TRAINING_MODE]}")

## 5. Imports & Helpers

In [ ]:
import math
import json
import shutil
import gc
import types as _types
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    DataCollatorForLanguageModeling,
)
from peft import (
    LoraConfig,
    PromptTuningConfig,
    PromptTuningInit,
    get_peft_model,
    PeftModel,
    TaskType,
    prepare_model_for_kbit_training,
)
from datasets import Dataset

print(f"PyTorch {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# ── Prompt formatting ─────────────────────────────────────────────────────────

def format_prompt(text, label=None, include_system_prompt=True):
    """Format a prompt for training or inference."""
    prefix = SYSTEM_PROMPT if include_system_prompt else ""
    user_prompt = f"Text: {text}"
    if label is not None:
        return f"{prefix}\n\nUser: {user_prompt}\nAssistant: {label}"
    return f"{prefix}\n\nUser: {user_prompt}\nAssistant:"


# ── Data loading ──────────────────────────────────────────────────────────────

def load_train_val_data(path=TRAIN_DATA_PATH):
    """Load training xlsx and split 80/20 into train/validation."""
    xl = pd.ExcelFile(path)
    print(f"Sheets: {xl.sheet_names}")
    all_t, all_tl, all_v, all_vl = [], [], [], []
    for sheet in xl.sheet_names:
        df = pd.read_excel(path, sheet_name=sheet)
        tt, vt, tl, vl = train_test_split(
            df['text'].values, df['label'].values,
            test_size=0.2, random_state=42)
        all_t.extend(tt); all_tl.extend(tl)
        all_v.extend(vt); all_vl.extend(vl)
        print(f"  {sheet}: {len(tt)} train, {len(vt)} val")
    print(f"Total: {len(all_t)} train, {len(all_v)} val")
    return np.array(all_t), np.array(all_tl), np.array(all_v), np.array(all_vl)


def load_test_data(path=TEST_DATA_PATH):
    """Load test xlsx for evaluation."""
    xl = pd.ExcelFile(path)
    texts, labels, sheets = [], [], []
    for sheet in xl.sheet_names:
        df = pd.read_excel(path, sheet_name=sheet)
        texts.extend(df['text'].values)
        labels.extend(df['label'].values)
        sheets.extend([sheet] * len(df))
    print(f"Test set: {len(texts)} examples")
    return np.array(texts), np.array(labels), np.array(sheets)


# ── Dataset preparation ───────────────────────────────────────────────────────

def prepare_dataset(texts, labels, tokenizer, include_system_prompt=True):
    """Tokenize texts into a HuggingFace Dataset."""
    input_ids_list, attention_mask_list = [], []
    for i in range(0, len(texts), 10):
        batch = [format_prompt(t, l, include_system_prompt)
                 for t, l in zip(texts[i:i+10], labels[i:i+10])]
        tok = tokenizer(batch, truncation=True, max_length=MAX_LENGTH,
                        padding=False, return_tensors=None)
        input_ids_list.extend(tok['input_ids'])
        attention_mask_list.extend(tok['attention_mask'])
    token_type_ids_list = [[0] * len(ids) for ids in input_ids_list]
    ds = Dataset.from_dict({
        'input_ids': input_ids_list,
        'attention_mask': attention_mask_list,
        'token_type_ids': token_type_ids_list,
    })
    print(f"Dataset: {len(ds)} examples")
    return ds

In [ ]:
# ── Model loading ────────────────────────────────────────────────────────────

def load_base_model():
    """Load base MamayLM with 8-bit quantization."""
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    try:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME, load_in_8bit=True, device_map="auto",
            torch_dtype=torch.bfloat16, low_cpu_mem_usage=True)
        print("Loaded with 8-bit quantization")
    except Exception as e:
        print(f"8-bit not available ({e}), using bfloat16")
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME, device_map="cuda:0",
            torch_dtype=torch.bfloat16, low_cpu_mem_usage=True)
    return tokenizer, model


def setup_lora(model):
    """Apply LoRA adapters."""
    model = prepare_model_for_kbit_training(model)
    lora_config = LoraConfig(
        r=LORA_RANK, lora_alpha=LORA_ALPHA,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_dropout=LORA_DROPOUT, bias="none",
        task_type=TaskType.CAUSAL_LM, inference_mode=False)
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model


def patch_peft_forward_for_gemma3(peft_model, num_virtual_tokens):
    """Monkey-patch PEFT forward to pass token_type_ids for Gemma3."""
    def _patched_forward(self, input_ids=None, attention_mask=None,
                         inputs_embeds=None, labels=None,
                         output_attentions=None, output_hidden_states=None,
                         return_dict=None, task_ids=None, **kwargs):
        batch_size = (input_ids.shape[0] if input_ids is not None
                      else inputs_embeds.shape[0])
        token_type_ids = kwargs.pop("token_type_ids", None)
        prompts = self.get_prompt(batch_size=batch_size, task_ids=task_ids)
        if inputs_embeds is None:
            inputs_embeds = self.word_embeddings(input_ids)
        inputs_embeds = torch.cat(
            (prompts.to(inputs_embeds.dtype), inputs_embeds), dim=1)
        if attention_mask is not None:
            prefix_mask = torch.ones(
                batch_size, num_virtual_tokens,
                device=attention_mask.device, dtype=attention_mask.dtype)
            attention_mask = torch.cat((prefix_mask, attention_mask), dim=1)
        if token_type_ids is not None:
            prefix_ttids = torch.zeros(
                batch_size, num_virtual_tokens,
                device=token_type_ids.device, dtype=token_type_ids.dtype)
            kwargs["token_type_ids"] = torch.cat(
                (prefix_ttids, token_type_ids), dim=1)
        if labels is not None:
            prefix_labels = torch.full(
                (batch_size, num_virtual_tokens), -100,
                device=labels.device, dtype=labels.dtype)
            labels = torch.cat((prefix_labels, labels), dim=1)
        return self.base_model(
            inputs_embeds=inputs_embeds, labels=labels,
            attention_mask=attention_mask,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict, **kwargs)
    peft_model.forward = _types.MethodType(_patched_forward, peft_model)


def setup_prompt_tuning(model, tokenizer):
    """Apply prompt-tuning adapter."""
    system_tokens = tokenizer(SYSTEM_PROMPT, return_tensors="pt").input_ids
    num_virtual_tokens = system_tokens.shape[1]
    print(f"Using {num_virtual_tokens} virtual tokens")
    pt_config = PromptTuningConfig(
        task_type=TaskType.CAUSAL_LM,
        prompt_tuning_init=PromptTuningInit.TEXT,
        prompt_tuning_init_text=SYSTEM_PROMPT,
        num_virtual_tokens=num_virtual_tokens,
        tokenizer_name_or_path=MODEL_NAME)
    model = get_peft_model(model, pt_config)
    model.print_trainable_parameters()
    patch_peft_forward_for_gemma3(model, num_virtual_tokens)
    return model, num_virtual_tokens

In [ ]:
# ── Metrics & callbacks ───────────────────────────────────────────────────────

def _preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits.argmax(dim=-1)


def _compute_token_accuracy(eval_preds):
    preds, labels = eval_preds
    preds = preds[:, :-1]
    labels = labels[:, 1:]
    mask = labels != -100
    correct = (preds[mask] == labels[mask]).sum()
    total = mask.sum()
    return {'accuracy': round(float(correct) / float(total), 4) if total > 0 else 0.0}


# ── Checkpoint backup ─────────────────────────────────────────────────────────

def backup_to_drive(local_dir, iteration):
    """Copy checkpoint from local storage to Google Drive."""
    backup_name = f"{TRAINING_MODE}_iter{iteration:02d}"
    dest = os.path.join(DRIVE_BACKUP_DIR, backup_name)
    if os.path.exists(dest):
        shutil.rmtree(dest)
    shutil.copytree(local_dir, dest)
    # Also keep a "latest" symlink-like copy
    latest = os.path.join(DRIVE_BACKUP_DIR, f"{TRAINING_MODE}_latest")
    if os.path.exists(latest):
        shutil.rmtree(latest)
    shutil.copytree(local_dir, latest)
    print(f"✓ Backed up to Drive: {dest}")
    print(f"✓ Updated latest:     {latest}")


def get_latest_checkpoint():
    """Find the latest checkpoint on Drive for the current mode."""
    latest = os.path.join(DRIVE_BACKUP_DIR, f"{TRAINING_MODE}_latest")
    if os.path.exists(latest):
        return latest
    return None


def restore_from_drive():
    """Restore latest checkpoint from Drive to local dir."""
    src = get_latest_checkpoint()
    if src is None:
        print("No checkpoint found on Drive")
        return None
    if os.path.exists(LOCAL_OUTPUT_DIR):
        shutil.rmtree(LOCAL_OUTPUT_DIR)
    shutil.copytree(src, LOCAL_OUTPUT_DIR)
    print(f"✓ Restored from: {src}")
    return LOCAL_OUTPUT_DIR


# ── Iteration state (persisted to Drive) ──────────────────────────────────────

STATE_FILE = os.path.join(DRIVE_BACKUP_DIR, "training_state.json")

def load_state():
    if os.path.exists(STATE_FILE):
        with open(STATE_FILE) as f:
            return json.load(f)
    return {"completed_iterations": 0, "total_epochs_trained": 0, "history": []}

def save_state(state):
    with open(STATE_FILE, 'w') as f:
        json.dump(state, f, indent=2)

print("Helpers loaded ✓")

## 6. Load Data

In [ ]:
train_texts, train_labels, val_texts, val_labels = load_train_val_data()

## 7. Run Training Iteration

Run the cell below **once per iteration**. It will:
1. Load the base model + resume adapter weights if a previous checkpoint exists
2. Train for `EPOCHS_PER_ITERATION` epochs
3. Save weights locally and backup to Google Drive
4. Free GPU memory

You can **re-run** this cell as many times as you need. Each run appends more training on top of the previous checkpoint.

In [ ]:
# ── Single training iteration ─────────────────────────────────────────────────

state = load_state()
iteration = state["completed_iterations"] + 1
print(f"═" * 60)
print(f"ITERATION {iteration}  (epochs {state['total_epochs_trained']+1}–"
      f"{state['total_epochs_trained']+EPOCHS_PER_ITERATION})")
print(f"═" * 60)

# 1) Load base model
tokenizer, base_model = load_base_model()

# 2) Apply PEFT / setup
include_system_prompt = True
lr = LEARNING_RATES[TRAINING_MODE]
use_gradient_checkpointing = False

if TRAINING_MODE == "lora":
    checkpoint_path = get_latest_checkpoint()
    if checkpoint_path:
        print(f"Resuming LoRA from: {checkpoint_path}")
        model = PeftModel.from_pretrained(base_model, checkpoint_path,
                                         is_trainable=True)
    else:
        print("Starting fresh LoRA training")
        model = setup_lora(base_model)

elif TRAINING_MODE == "prompt_tuning":
    include_system_prompt = False
    checkpoint_path = get_latest_checkpoint()
    if checkpoint_path:
        print(f"Resuming prompt-tuning from: {checkpoint_path}")
        model = PeftModel.from_pretrained(base_model, checkpoint_path,
                                         is_trainable=True)
        # Re-read num_virtual_tokens from config
        nvt = model.peft_config["default"].num_virtual_tokens
        patch_peft_forward_for_gemma3(model, nvt)
    else:
        print("Starting fresh prompt-tuning")
        model, _ = setup_prompt_tuning(base_model, tokenizer)

elif TRAINING_MODE == "full_finetune":
    use_gradient_checkpointing = True
    checkpoint_path = get_latest_checkpoint()
    if checkpoint_path:
        print(f"Resuming full finetune from: {checkpoint_path}")
        del base_model
        torch.cuda.empty_cache(); gc.collect()
        tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(
            checkpoint_path, device_map="cuda:0",
            torch_dtype=torch.bfloat16, low_cpu_mem_usage=True)
    else:
        print("Starting fresh full finetune")
        model = base_model
    model.gradient_checkpointing_enable()

# 3) Prepare datasets
train_dataset = prepare_dataset(train_texts, train_labels, tokenizer,
                                include_system_prompt)
val_dataset = prepare_dataset(val_texts, val_labels, tokenizer,
                              include_system_prompt)

# 4) Trainer
training_args = TrainingArguments(
    output_dir=LOCAL_OUTPUT_DIR,
    num_train_epochs=EPOCHS_PER_ITERATION,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=lr,
    warmup_steps=WARMUP_STEPS if iteration == 1 else 0,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    bf16=True,
    optim="adamw_torch",
    remove_unused_columns=False,
    report_to="none",
    gradient_checkpointing=use_gradient_checkpointing,
    max_grad_norm=1.0,
    dataloader_pin_memory=False,
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=_compute_token_accuracy,
    preprocess_logits_for_metrics=_preprocess_logits_for_metrics,
)

# 5) Train
print("\nTraining...")
train_result = trainer.train()

# 6) Save locally
print(f"\nSaving to {LOCAL_OUTPUT_DIR}...")
trainer.save_model(LOCAL_OUTPUT_DIR)
tokenizer.save_pretrained(LOCAL_OUTPUT_DIR)

# For prompt tuning, also save raw embeddings
if TRAINING_MODE == "prompt_tuning":
    try:
        emb = trainer.model.prompt_encoder["default"].embedding.weight.data
        torch.save(emb.cpu(), os.path.join(LOCAL_OUTPUT_DIR, "prompt_embeddings.pt"))
        print(f"Saved prompt embeddings: shape {emb.shape}")
    except Exception:
        pass

# 7) Backup to Drive
backup_to_drive(LOCAL_OUTPUT_DIR, iteration)

# 8) Update state
eval_metrics = trainer.evaluate()
state["completed_iterations"] = iteration
state["total_epochs_trained"] += EPOCHS_PER_ITERATION
state["history"].append({
    "iteration": iteration,
    "train_loss": round(train_result.metrics.get("train_loss", 0), 4),
    "eval_loss": round(eval_metrics.get("eval_loss", 0), 4),
    "eval_accuracy": round(eval_metrics.get("eval_accuracy", 0), 4),
    "timestamp": datetime.now().isoformat(),
})
save_state(state)

# 9) Cleanup
del trainer, model, train_dataset, val_dataset
if 'base_model' in dir():
    try:
        del base_model
    except Exception:
        pass
torch.cuda.empty_cache()
gc.collect()

# 10) Print summary
print(f"\n{'═' * 60}")
print(f"ITERATION {iteration} COMPLETE")
print(f"{'═' * 60}")
print(f"Total epochs trained: {state['total_epochs_trained']}")
for h in state["history"]:
    print(f"  iter {h['iteration']:2d}: train_loss={h['train_loss']:.4f}  "
          f"eval_loss={h['eval_loss']:.4f}  eval_acc={h['eval_accuracy']:.4f}")
print(f"\n→ Weights backed up to Drive. You can safely restart the runtime.")
print(f"→ Re-run this cell to continue training iteration {iteration + 1}.")

## 8. Training Progress

View all completed iterations and metrics.

In [ ]:
state = load_state()
print(f"Completed iterations: {state['completed_iterations']}")
print(f"Total epochs trained: {state['total_epochs_trained']}")
print(f"Mode: {TRAINING_MODE}")
print()

if state["history"]:
    print(f"{'Iter':>4}  {'Train Loss':>10}  {'Eval Loss':>10}  {'Eval Acc':>10}  {'Time'}")
    print("─" * 65)
    for h in state["history"]:
        print(f"{h['iteration']:4d}  {h['train_loss']:10.4f}  {h['eval_loss']:10.4f}  "
              f"{h['eval_accuracy']:10.4f}  {h.get('timestamp', 'N/A')}")
else:
    print("No training history yet.")

# List checkpoint files on Drive
print(f"\nCheckpoints on Drive ({DRIVE_BACKUP_DIR}):")
if os.path.exists(DRIVE_BACKUP_DIR):
    for item in sorted(os.listdir(DRIVE_BACKUP_DIR)):
        full = os.path.join(DRIVE_BACKUP_DIR, item)
        if os.path.isdir(full):
            size_mb = sum(f.stat().st_size for f in Path(full).rglob('*') if f.is_file()) / 1e6
            print(f"  📁 {item} ({size_mb:.0f} MB)")
        else:
            print(f"  📄 {item}")

## 9. Interactive Testing

Load the fine-tuned model and test it on individual phrases. Run the first cell to load the model, then use the second cell to test as many phrases as you want.

In [ ]:
# ── Load model for testing ────────────────────────────────────────────────────

# Use latest checkpoint from Drive (survives runtime restarts)
test_model_path = get_latest_checkpoint()
if test_model_path is None:
    test_model_path = LOCAL_OUTPUT_DIR

print(f"Loading model from: {test_model_path}")
print(f"Mode: {TRAINING_MODE}")

if TRAINING_MODE == "full_finetune":
    test_tokenizer = AutoTokenizer.from_pretrained(test_model_path)
    if test_tokenizer.pad_token is None:
        test_tokenizer.pad_token = test_tokenizer.eos_token
    test_model = AutoModelForCausalLM.from_pretrained(
        test_model_path, device_map="cuda:0",
        torch_dtype=torch.bfloat16, low_cpu_mem_usage=True)
else:
    # LoRA or prompt-tuning: load base + adapter
    test_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if test_tokenizer.pad_token is None:
        test_tokenizer.pad_token = test_tokenizer.eos_token
    test_base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, device_map="cuda:0",
        torch_dtype=torch.bfloat16, low_cpu_mem_usage=True)
    test_model = PeftModel.from_pretrained(test_base, test_model_path)
    if TRAINING_MODE == "lora":
        test_model = test_model.merge_and_unload()

test_model.eval()
print("Model loaded ✓")

In [ ]:
# ── Test a single phrase ──────────────────────────────────────────────────────
# Edit the text below and re-run this cell as many times as you want.

TEST_TEXT = "Вчора вночі було кілька <прильотів> по місту."

is_prompt_tuning = (TRAINING_MODE == "prompt_tuning")
prompt = format_prompt(TEST_TEXT, include_system_prompt=not is_prompt_tuning)
inputs = test_tokenizer(prompt, return_tensors="pt").to(test_model.device)

with torch.no_grad():
    outputs = test_model.generate(
        **inputs, max_new_tokens=10, do_sample=False,
        pad_token_id=test_tokenizer.eos_token_id)

result = test_tokenizer.decode(outputs[0], skip_special_tokens=True)
result = result[len(prompt):].strip()
prediction = 0
for ch in result:
    if ch == '1':
        prediction = 1
        break
    elif ch == '0':
        prediction = 0
        break

print(f"Text:       {TEST_TEXT}")
print(f"Raw output: {result}")
print(f"Prediction: {prediction} ({'Euphemism' if prediction == 1 else 'Not euphemism'})")

del inputs, outputs
torch.cuda.empty_cache()

In [ ]:
# ── Free test model when done ─────────────────────────────────────────────────
# Run this before starting the next training iteration to free GPU memory.

del test_model, test_tokenizer
if 'test_base' in dir():
    del test_base
torch.cuda.empty_cache()
gc.collect()
print("Test model freed ✓")

## 10. Evaluate on Test Set

Run full evaluation on `PETs_Ukr_Test.xlsx` with per-sheet metrics.

In [ ]:
# Load test data
test_texts, test_labels, test_sheet_names = load_test_data()

# Load model
eval_model_path = get_latest_checkpoint() or LOCAL_OUTPUT_DIR
print(f"Evaluating model from: {eval_model_path}")

is_pt = (TRAINING_MODE == "prompt_tuning")

if TRAINING_MODE == "full_finetune":
    eval_tokenizer = AutoTokenizer.from_pretrained(eval_model_path)
    if eval_tokenizer.pad_token is None:
        eval_tokenizer.pad_token = eval_tokenizer.eos_token
    eval_model = AutoModelForCausalLM.from_pretrained(
        eval_model_path, device_map="cuda:0",
        torch_dtype=torch.bfloat16, low_cpu_mem_usage=True)
else:
    eval_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if eval_tokenizer.pad_token is None:
        eval_tokenizer.pad_token = eval_tokenizer.eos_token
    eval_base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, device_map="cuda:0",
        torch_dtype=torch.bfloat16, low_cpu_mem_usage=True)
    eval_model = PeftModel.from_pretrained(eval_base, eval_model_path)
    if TRAINING_MODE == "lora":
        eval_model = eval_model.merge_and_unload()

eval_model.eval()

# Predict
print(f"\nRunning predictions on {len(test_texts)} examples...")
predictions = []
for i, text in enumerate(test_texts):
    prompt = format_prompt(text, include_system_prompt=not is_pt)
    inputs = eval_tokenizer(prompt, return_tensors="pt").to(eval_model.device)
    with torch.no_grad():
        outputs = eval_model.generate(
            **inputs, max_new_tokens=10, do_sample=False,
            pad_token_id=eval_tokenizer.eos_token_id)
    result = eval_tokenizer.decode(outputs[0], skip_special_tokens=True)
    result = result[len(prompt):].strip()
    pred = 0
    for ch in result:
        if ch == '1': pred = 1; break
        elif ch == '0': pred = 0; break
    predictions.append(pred)
    del inputs, outputs
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(test_texts)}")
        torch.cuda.empty_cache()

predictions = np.array(predictions)

# Overall metrics
accuracy = accuracy_score(test_labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(
    test_labels, predictions, average='binary', zero_division=0)
cm = confusion_matrix(test_labels, predictions)

print(f"\n{'═' * 60}")
print(f"OVERALL RESULTS")
print(f"{'═' * 60}")
print(f"Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"\nConfusion Matrix:")
print(f"  TN: {cm[0,0]}  FP: {cm[0,1]}")
print(f"  FN: {cm[1,0]}  TP: {cm[1,1]}")

# Per-sheet metrics
print(f"\n{'═' * 60}")
print(f"PER-SHEET RESULTS")
print(f"{'═' * 60}")
seen = set()
unique_sheets = [s for s in test_sheet_names if s not in seen and not seen.add(s)]
for sheet in unique_sheets:
    mask = test_sheet_names == sheet
    sl, sp = test_labels[mask], predictions[mask]
    cm_s = confusion_matrix(sl, sp, labels=[0, 1])
    tp, tn, fp, fn = int(cm_s[1,1]), int(cm_s[0,0]), int(cm_s[0,1]), int(cm_s[1,0])
    n = int(mask.sum())
    acc = (tp+tn)/n if n else 0
    prec = tp/(tp+fp) if (tp+fp) else 0
    rec = tp/(tp+fn) if (tp+fn) else 0
    f1_s = 2*prec*rec/(prec+rec) if (prec+rec) else 0
    print(f"\n{sheet}: n={n}  acc={acc:.3f}  prec={prec:.3f}  rec={rec:.3f}  f1={f1_s:.3f}")
    print(f"  TP={tp} TN={tn} FP={fp} FN={fn}")

# Cleanup
del eval_model, eval_tokenizer
if 'eval_base' in dir():
    del eval_base
torch.cuda.empty_cache()
gc.collect()
print(f"\n✓ Evaluation complete")

## 11. Download Model from Drive

Download the final fine-tuned model to your local machine.

In [ ]:
# Zip and download the latest checkpoint
latest = get_latest_checkpoint()
if latest:
    archive = shutil.make_archive("/content/mamaylm_finetuned", 'zip', latest)
    files.download(archive)
    print(f"Downloading {archive}")
else:
    print("No checkpoint found")

## 12. Reset Training State

⚠️ Run this only if you want to start training from scratch.

In [ ]:
# ⚠️ DANGER: Uncomment to reset all training state and delete checkpoints
# import shutil, os
# if os.path.exists(DRIVE_BACKUP_DIR):
#     shutil.rmtree(DRIVE_BACKUP_DIR)
#     os.makedirs(DRIVE_BACKUP_DIR)
# if os.path.exists(LOCAL_OUTPUT_DIR):
#     shutil.rmtree(LOCAL_OUTPUT_DIR)
# print("All training state and checkpoints deleted.")